# FinBERT — FinancialPhraseBank Sentiment Evaluation

**Flow**

1. Load **FinancialPhraseBank** (75% agreement) — stratified **80/20** (train/test) split.
2. Run pretrained **FinBERT** on the test set (no fine-tuning).
3. Report metrics and save the model locally.


In [9]:
!nvidia-smi

Tue Apr  7 17:26:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.44.01              Driver Version: 590.44.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
| 37%   32C    P8             29W /  350W |    7392MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# Fix torchvision/PyTorch CUDA version mismatch.
# FinBERT is text-only — torchvision is not needed.
# Run once, then restart the kernel.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchvision", "-y"], check=False)

CompletedProcess(args=['/common/home/users/w/wlcheng.2023/jupyterlab-venv-pytorch-240/bin/python3', '-m', 'pip', 'uninstall', 'torchvision', '-y'], returncode=0)

In [11]:
import io
import pandas as pd
import requests
from sklearn.model_selection import train_test_split

SEED = 42
LABELS = ["negative", "neutral", "positive"]
LABEL_SET = set(LABELS)
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

FPB_URL = (
    "https://raw.githubusercontent.com/maxwellsarpong/"
    "NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt"
)

resp = requests.get(FPB_URL, timeout=60)
resp.raise_for_status()
fpb_df = pd.read_csv(
    io.StringIO(resp.text),
    sep="@",
    header=None,
    names=["sentence", "label_text"],
    engine="python",
    encoding="latin-1",
)
fpb_df["sentence"] = fpb_df["sentence"].str.strip()
fpb_df["label_text"] = fpb_df["label_text"].str.strip().str.lower()
fpb_df = fpb_df[fpb_df["label_text"].isin(LABEL_SET)].reset_index(drop=True)
fpb_df["label"] = fpb_df["label_text"].map(LABEL2ID)
print(f"FPB: {len(fpb_df)} samples")
print(fpb_df["label_text"].value_counts())

train_df, test_df = train_test_split(
    fpb_df, test_size=0.20, random_state=SEED, stratify=fpb_df["label_text"]
)
print(f"\nTrain {len(train_df)} | Test {len(test_df)}")

FPB: 3453 samples
label_text
neutral     2146
positive     887
negative     420
Name: count, dtype: int64

Train 2762 | Test 691


In [12]:
import torch


def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x if x in LABEL_SET else "neutral" for x in pred_labels]
    return {
        "macro_f1": f1_score(
            true_labels, pred_labels, labels=LABELS, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            true_labels, pred_labels, labels=LABELS, average="weighted", zero_division=0
        ),
        "accuracy": accuracy_score(true_labels, pred_labels),
        "mcc": matthews_corrcoef(true_labels, pred_labels),
        "f1_negative": f1_score(
            true_labels,
            pred_labels,
            labels=["negative"],
            average="macro",
            zero_division=0,
        ),
        "f1_neutral": f1_score(
            true_labels,
            pred_labels,
            labels=["neutral"],
            average="macro",
            zero_division=0,
        ),
        "f1_positive": f1_score(
            true_labels,
            pred_labels,
            labels=["positive"],
            average="macro",
            zero_division=0,
        ),
        "report": classification_report(
            true_labels, pred_labels, labels=LABELS, zero_division=0
        ),
    }


def print_metrics(title, m):
    print("=" * 70, title, "=" * 70, m["report"], sep="\n")
    for k in (
        "macro_f1",
        "weighted_f1",
        "accuracy",
        "mcc",
        "f1_negative",
        "f1_neutral",
        "f1_positive",
    ):
        print(f"  {k}: {m[k]:.4f}")

## Pretrained FinBERT — Evaluation on FPB Test Set


In [13]:
import gc
import json
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    matthews_corrcoef,
)

MODEL_NAME = "ProsusAI/finbert"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_BASE = Path("genai_grp_project")
CKPT_BASE.mkdir(parents=True, exist_ok=True)

print(f"Model : {MODEL_NAME}")
print(f"Device: {DEVICE}")

Model : ProsusAI/finbert
Device: cuda


In [14]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

base_tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(DEVICE)


# FinBERT uses its own label ordering; remap to our standard
def remap_finbert_pred(p):
    label = base_model.config.id2label.get(p, "neutral").lower()
    return label if label in LABEL_SET else "neutral"


base_preds_raw = []
sents = test_df["sentence"].tolist()
with torch.inference_mode():
    for i in range(0, len(sents), 128):
        enc = base_tok(
            sents[i : i + 128],
            truncation=True,
            padding=True,
            max_length=128,
            return_tensors="pt",
        ).to(DEVICE)
        logits = base_model(**enc).logits
        base_preds_raw.extend(logits.argmax(-1).tolist())

base_preds = [remap_finbert_pred(p) for p in base_preds_raw]
base_metrics = evaluate_predictions(test_df["label_text"].tolist(), base_preds)
print_metrics("FinBERT (pretrained) — FPB Test Set", base_metrics)

# ─── Save model ───────────────────────────────────────────────────────────────
SAVED_DIR = CKPT_BASE / "finbert_saved"
base_model.save_pretrained(str(SAVED_DIR))
base_tok.save_pretrained(str(SAVED_DIR))

(SAVED_DIR / "inference_config.json").write_text(
    json.dumps(
        {
            "model_type": "sequence_classification",
            "model_name_or_path": str(SAVED_DIR),
            "base_model": MODEL_NAME,
            "labels": LABELS,
            "label2id": LABEL2ID,
            "id2label": {str(k): v for k, v in ID2LABEL.items()},
            "test_macro_f1": round(base_metrics["macro_f1"], 4),
        },
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Model saved → {SAVED_DIR}")
print(f"inference_config.json → {SAVED_DIR}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT (pretrained) — FPB Test Set
              precision    recall  f1-score   support

    negative       0.89      0.98      0.93        84
     neutral       1.00      0.95      0.97       429
    positive       0.90      0.97      0.93       178

    accuracy                           0.96       691
   macro avg       0.93      0.96      0.94       691
weighted avg       0.96      0.96      0.96       691

  macro_f1: 0.9447
  weighted_f1: 0.9557
  accuracy: 0.9551
  mcc: 0.9192
  f1_negative: 0.9318
  f1_neutral: 0.9701
  f1_positive: 0.9322


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved → genai_grp_project/finbert_saved
inference_config.json → genai_grp_project/finbert_saved


In [15]:
cols = [
    "macro_f1",
    "weighted_f1",
    "accuracy",
    "mcc",
    "f1_negative",
    "f1_neutral",
    "f1_positive",
]
display(
    pd.DataFrame(
        [
            {
                "model": "FinBERT (pretrained)",
                **{k: v for k, v in base_metrics.items() if isinstance(v, float)},
            }
        ]
    )[["model"] + cols]
)

,model,macro_f1,weighted_f1,accuracy,mcc,f1_negative,f1_neutral,f1_positive
0,FinBERT (pretrained),0.944733,0.955716,0.955137,0.919247,0.931818,0.970131,0.932249


In [16]:
del base_model, base_tok
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Done.")

Done.
